In [1]:
# Pin here for fix of CPU offloading bug; implemented in docker /uv
#%pip install "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1" "vllm<0.22.0"

In [2]:
from experiments_pretrained import *
from data import *
from config_record_activations import *
import pickle

/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"

In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Get data: general prompts
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
prompts = format_prompts_mmlu(dataset)

Streaming cais/mmlu (all) (samples: 100)...


In [6]:
# Get model
model, tokenizer = load_model(model_id, enable_bnb=True)
probe = MoEProbeMistral(model)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

load_tokenizer: no pad token defined for mistralai/Mixtral-8x7B-Instruct-v0.1, using eos_token ('</s>') as pad_token.
MoEHook: Scanning model for routers...
MoEHook: Attached probes to 32 router layers.
MoEHook: Model has 32 routers each with 8 experts and selects k=2 at each layer.


In [7]:
# Record activations over generalized MMLU questions
results = get_activations_mmlu(model, tokenizer, dataset, probe=probe, max_new_tokens=max_new_tokens, batch_size=batch_size)
#results = get_activations_mmlu(model, tokenizer, dataset, probe=probe, max_new_tokens=max_new_tokens, batch_size=1)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Generating responses 1-32/100...


/home/dylan/.local/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Batch inference took 76.630s (2.395s/sample).
Generating responses 33-64/100...
Batch inference took 76.859s (2.402s/sample).
Generating responses 65-96/100...
Batch inference took 76.414s (2.388s/sample).
Generating responses 97-100/100...
Batch inference took 32.725s (8.181s/sample).


In [8]:
# Save results
with open(results_file, 'wb') as file:
    pickle.dump(results, file)

In [11]:
with open("results_nonbatched.pkl", 'rb') as file:
    results_nonbatched = pickle.load(file)
with open("results_batched.pkl", 'rb') as file:
    results_batched = pickle.load(file)

prompt_id = 1
print(results_nonbatched[prompt_id]['prompt'])
print(results_nonbatched[prompt_id]['subject'])
print(results_nonbatched[prompt_id]['probs'].shape)
print(results_nonbatched[prompt_id]['active_experts'][0, :, 1])
print(results_batched[prompt_id]['prompt'])
print(results_batched[prompt_id]['subject'])
print(results_batched[prompt_id]['probs'].shape)
print(results_batched[prompt_id]['active_experts'][0, :, 1])

Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the index of <p> in S_5.
abstract_algebra
torch.Size([1, 253, 8, 32])
tensor([[1, 4, 1,  ..., 3, 6, 7],
        [1, 4, 1,  ..., 3, 6, 7],
        [3, 5, 1,  ..., 0, 5, 5],
        ...,
        [7, 7, 4,  ..., 7, 4, 1],
        [4, 0, 2,  ..., 4, 4, 6],
        [0, 0, 6,  ..., 4, 1, 4]])
Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the index of <p> in S_5.
abstract_algebra
torch.Size([253, 8, 32])
tensor([3, 4])
